# EDA: Aave V3 vs Compound V3 — USDC Borrow Rates
Exploratory analysis of historical borrow APY data pulled from the DefiLlama API.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_all

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

## 1. Overview

In [ ]:
df = load_all()

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
print('Columns & dtypes:')
for col, dtype in df.dtypes.items():
    print(f'  {col:<20} {dtype}')
print()
df.head(10)

## 2. Target Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Record count per protocol (class distribution)
counts = df['project'].value_counts()
axes[0].bar(counts.index, counts.values, color=sns.color_palette('muted', len(counts)))
axes[0].set_title('Record Count by Protocol', fontsize=13)
axes[0].set_xlabel('Protocol')
axes[0].set_ylabel('Number of Records')
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 5, f'{val:,}', ha='center', fontsize=10)

# apyBase distribution per protocol
for project, grp in df.groupby('project'):
    axes[1].hist(grp['apyBase'].dropna(), bins=50, alpha=0.6, label=project)
axes[1].set_title('apyBase Distribution by Protocol', fontsize=13)
axes[1].set_xlabel('Borrow APY (%)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.suptitle('Target Analysis', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Align on overlapping dates for fair comparison
aave = df[df['project'] == 'aave-v3']
comp = df[df['project'] == 'compound-v3']

aave_d = aave.set_index(aave['timestamp'].dt.date)
comp_d = comp.set_index(comp['timestamp'].dt.date)
common_dates = aave_d.index.intersection(comp_d.index)

aave_base   = aave_d.loc[common_dates, 'apyBase']
comp_base   = comp_d.loc[common_dates, 'apyBase']
comp_reward = comp_d.loc[common_dates, 'apyReward'].fillna(0)
comp_net    = comp_base - comp_reward  # reward is paid to borrowers, reducing net cost

# Summary table
comparison = pd.DataFrame({
    'Aave apyBase':         [aave_base.mean()],
    'Compound apyBase':     [comp_base.mean()],
    'Compound apyReward':   [comp_reward.mean()],
    'Compound net cost':    [comp_net.mean()],
}, index=['Mean APY (%)'])
print(f'Overlapping period: {len(common_dates)} days\n')
print(comparison.round(4).to_string())
print(f'\nAave base > Compound base on {(aave_base.values > comp_base.values).mean()*100:.1f}% of days')
print(f'Aave base > Compound net on  {(aave_base.values > comp_net.values).mean()*100:.1f}% of days')

# Plot: Aave apyBase vs Compound apyBase vs Compound net cost
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

idx = pd.to_datetime(list(common_dates))
axes[0].plot(idx, aave_base.values, label='Aave apyBase',        alpha=0.8, linewidth=0.8)
axes[0].plot(idx, comp_base.values, label='Compound apyBase',     alpha=0.8, linewidth=0.8)
axes[0].plot(idx, comp_net.values,  label='Compound net cost',    alpha=0.8, linewidth=0.8, linestyle='--')
axes[0].set_title('Aave vs Compound: Base Rate and Net Borrow Cost', fontsize=12)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('APY (%)')
axes[0].legend()

axes[1].hist(aave_base.values, bins=50, alpha=0.6, label='Aave apyBase')
axes[1].hist(comp_base.values, bins=50, alpha=0.6, label='Compound apyBase')
axes[1].hist(comp_net.values,  bins=50, alpha=0.4, label='Compound net cost', linestyle='--', edgecolor='grey')
axes[1].set_title('Distribution: Aave Base vs Compound Base vs Compound Net', fontsize=12)
axes[1].set_xlabel('APY (%)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.suptitle('Reward-Adjusted Rate Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Missing Values

In [ ]:
null_matrix = df.isnull()
null_pct = (null_matrix.mean() * 100).round(1)

print('Null rates per column:')
for col, pct in null_pct.items():
    bar = '█' * int(pct / 5)
    print(f'  {col:<20} {pct:5.1f}%  {bar}')
print()

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    null_matrix.T,
    ax=ax,
    cbar=True,
    cmap='Blues',
    yticklabels=df.columns,
    xticklabels=False,
)
ax.set_title('Missing Value Heatmap (blue = null)', fontsize=13)
ax.set_xlabel('Records (time →)')
ax.set_ylabel('Column')
plt.tight_layout()
plt.show()

## 4. Feature Distributions

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()

n_cols = 3
n_rows = 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=40, color=sns.color_palette('muted')[i % 6], edgecolor='white')
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

# Hide unused subplots
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions', fontsize=15)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    ax=ax,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8},
)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Features vs Target

In [ ]:
# Top 3 numeric features most correlated with apyBase
if 'apyBase' in numeric_cols:
    target = 'apyBase'
    feature_cols = [c for c in numeric_cols if c != target]
    correlations = df[feature_cols].corrwith(df[target]).abs().sort_values(ascending=False)
    top_features = correlations.head(3).index.tolist()
else:
    top_features = numeric_cols[:3]
    target = numeric_cols[0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, feat in enumerate(top_features):
    sns.boxplot(
        data=df,
        x='project',
        y=feat,
        ax=axes[i],
        palette='muted',
    )
    axes[i].set_title(f'{feat} by Protocol', fontsize=11)
    axes[i].set_xlabel('Protocol')
    axes[i].set_ylabel(feat)

plt.suptitle('Feature Distributions by Protocol (Top 3 correlated with apyBase)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Key Findings

- **Compound is cheaper once rewards are accounted for.** COMP rewards are paid to borrowers, reducing their net cost. Compound's net borrow cost (apyBase − apyReward) averages **4.55%** vs Aave's **4.87%** — a 32 bps advantage for Compound. Aave is the more expensive protocol on **55% of days** once rewards are factored in, up from 45% on a base-vs-base comparison. **Rule: compare Aave's `apyBase` against Compound's `apyBase − apyReward` for a fair effective-cost comparison.**

- **Rates broadly co-move but are not in lockstep.** The mean spread (Aave base − Compound net) is +0.32% with a std of 2.81% and a range of −16.3% to +54.2%. The protocols track each other over long horizons but diverge sharply during demand shocks.

- **Aave V3 is more volatile.** Aave's `apyBase` std (3.45%) exceeds Compound's (2.90%), and Aave's max spike (56.7%) dwarfs Compound's (22.4%). Consistent with Aave holding ~6.6× more TVL ($397M vs $60M avg) — larger pools absorb bigger demand events as sharper rate moves.

- **`apyReward` is Compound-only and structurally null for Aave.** All 1,163 Aave records have null `apyReward`. It is the only column with missing data and is not missing at random — do not impute. Encode as a protocol-level feature or drop in favour of the net cost derived column.

- **`apyBase` is right-skewed with extreme spikes.** Both protocols have spike episodes (Aave: 65 days, Compound: 73 days above 10%). This will dominate error metrics in regression models — consider log-transforming the target or training separate spike/non-spike regime models.